In [1]:
import os
from typing import TypedDict, Annotated, Sequence

from dotenv import load_dotenv, find_dotenv
from langchain_core.messages import BaseMessage  # The foundational class for all message types in LangGraph
from langchain_core.messages import SystemMessage  # Message for providing instructions to the LLM
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages  # a reducer function; a rule that controls how updates from nodes are combined into the existing state. This lets us do a "deep append" of messages to the langgraph state
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
import wikipedia

load_dotenv(os.path.join("..", ".env"), override=True)

True

In [2]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [3]:
@tool
def calculator(expression: str) -> str:
    """Evaluates a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def search_wikipedia(query: str) -> str:
    """Simulates a Wikipedia lookup."""
    return wikipedia.summary(query, sentences=2)

In [4]:
tools = [calculator, search_wikipedia]
model = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

In [6]:
def agent_node(state: AgentState) -> AgentState:
    """
    This node will call the model and return the response.
    """
    system_prompt = SystemMessage(content="""
    You are a helpful assistant with various tools at your disposal. Please use the tools to answer my questions to the best of your ability.
    """)
    response = model.invoke([system_prompt] + state['messages'])

    return {"messages": [response]}

def is_done(state: AgentState) -> bool:
    """
    This node will determine if the agent should continue.
    """
    messages = state['messages']
    last_message = messages[-1]

    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

def print_stream(stream):
    for s in stream:
        message = s['messages'][-1]
        message.pretty_print()

In [7]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)

tool_node = ToolNode(tools=tools)
builder.add_node("tools", tool_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    is_done,
    {
        "end": END,
        "continue": "tools"
    }
)
builder.add_edge("tools", "agent")
agent = builder.compile()

In [8]:
inputs = {"messages": [("user", "What is the population of France plus Germany?")]}
print_stream(agent.stream(inputs, stream_mode="values"))

================================ Human Message =================================

What is the population of France plus Germany?
================================== Ai Message ==================================
Tool Calls:
  search_wikipedia (call_yR3NVuwfnrQ2tpA0eo8tpqSr)
 Call ID: call_yR3NVuwfnrQ2tpA0eo8tpqSr
  Args:
    query: Population of France 2023
  search_wikipedia (call_k02CEIGXSzzP4STpIfzzdSHm)
 Call ID: call_k02CEIGXSzzP4STpIfzzdSHm
  Args:
    query: Population of Germany 2023
================================= Tool Message =================================
Name: search_wikipedia

The demography of Germany is monitored by the Statistisches Bundesamt (Federal Statistical Office of Germany). According to the most recent data, Germany's population is 83,577,140 (31 December 2024) making it the most populous country in the European Union and the nineteenth-most populous country in the world.
================================== Ai Message ==================================
Tool C